# 필요 라이브러리 및 함수 정의

In [1]:
import redis
import time
import base64
import json
import csv
import random
import statistics

import numpy as np
import pandas as pd

In [2]:
def sliding_window(data, window_size, step):
    for start_row in range(0, len(data) - window_size + 1, step):
        yield data[start_row:start_row + window_size]        

# 입력 데이터 정의

In [3]:
#file = input()

In [4]:
#test = pd.read_csv(file,header=None)

In [5]:
test = pd.read_csv("text_1.csv", header=None)
#test = pd.read_csv("text_2.csv", header=None)
#test = pd.read_csv("text_3.csv", header=None)
test = test.transpose()
#test

# 데이터 전처리

In [6]:
#차후 변수 동적처리
humid = []
pm10 = []
pm25 = []
temp = []

for i in range(len(test)):
    val = test.values[i][0]
    val = val.replace("'", "")
    val = val.replace("b","",1)
    json_val = json.loads(val)
    
    res_payload = json_val['Payload']
    dec_res = base64.b64decode(res_payload)
    dec_res = dec_res.decode("UTF-8")
    str_test = dec_res.replace("'","\"")
    json_data = json.loads(str_test)
    
    humid.append(json_data['event']['readings'][0]['objectValue']['humidity'])
    pm10.append(json_data['event']['readings'][0]['objectValue']['pm10'])
    pm25.append(json_data['event']['readings'][0]['objectValue']['pm25'])
    temp.append(json_data['event']['readings'][0]['objectValue']['temperature'])

# 변수별 차분 계산

In [7]:
#절대값 차분
diff_humid = []
diff_pm10 = []
diff_pm25 = []
diff_temp = []

for i in range(len(humid)-1):
    diff_humid.append(abs(humid[i+1]-humid[i]))
    diff_pm10.append(abs(pm10[i+1]-pm10[i]))
    diff_pm25.append(abs(pm25[i+1]-pm25[i]))
    diff_temp.append(abs(temp[i+1]-temp[i]))

# 감소율을 위한 최적값 계산

In [8]:
while True :
    window_size = random.randint(1,100)
    step = random.randint(1,10)
    weight = random.uniform(0.1,0.3)

    humid_window = list(sliding_window(diff_humid,window_size,step))
    pm10_window = list(sliding_window(diff_pm10,window_size,step))
    pm25_window = list(sliding_window(diff_pm25,window_size,step))
    temp_window = list(sliding_window(diff_temp,window_size,step))

    avdf_humid = statistics.mean(humid_window[random.randint(1,len(humid_window)-1)])
    avdf_pm10 = statistics.mean(pm10_window[random.randint(1,len(pm10_window)-1)])
    avdf_pm25 = statistics.mean(pm25_window[random.randint(1,len(pm25_window)-1)])
    avdf_temp = statistics.mean(temp_window[random.randint(1,len(temp_window)-1)])

    cnt_val = []
    for i in range(len(diff_humid)):
        if diff_humid[i] < avdf_humid + weight and diff_pm10[i] < avdf_pm10 + weight and diff_pm25[i] < avdf_pm25 + weight and diff_temp[i] < avdf_temp + weight:
            cnt_val.append(i)
    
    if 100<len(cnt_val)<120:
        
        #print("윈도우 사이즈 : ", window_size)
        #print("스텝 사이즈 : ", step)
        #print("가중치 : ", weight)
        
        print("입력 트래픽 수 :", len(humid))
        print("출력 트래픽 수 : ", len(humid)-len(cnt_val))
        print("감소율 : ", round((len(humid)-(len(humid)-len(cnt_val)))/len(humid) * 100, 2), "%")
        break;

입력 트래픽 수 : 1000
출력 트래픽 수 :  895
감소율 :  10.5 %


In [10]:
avdf_humid

1.6666666666666667